In [1]:
import sys
using_colab = 'google.colab' in sys.modules
print(f"Is using Colab: {using_colab}")

Is using Colab: False


In [2]:
if using_colab:
    import subprocess
    import os
    from google.colab import drive
    drive.mount('/content/drive')
    os.chdir('/content/drive/MyDrive/Thesis_Workspace')
    print(os.listdir('.'))
    subprocess.run(['pip', 'install', 'ultralytics', 'lap'])

In [3]:
from IPython.display import clear_output
from pathlib import Path
from ultralytics import YOLO
from ultralytics.utils.metrics import box_iou

import csv
import cv2
import numpy as np
import os
import pandas as pd
import shutil
import sys
import time
import torch

In [4]:
using_gpu = None
if torch.cuda.is_available():
    print(True)
    print(torch.cuda.current_device())
    print(torch.cuda.get_device_name(0))
    using_gpu=0
else:
    print(False)
    print("No GPU available")
    using_gpu="cpu"

True
0
NVIDIA GeForce RTX 4050 Laptop GPU


In [5]:
### Models
detect_model_name = "models/detect/best_08222026.pt"
pose_model_name = "models/pose/yolo11x-pose.pt"
custom_tracker = "trackers/custom_botsort.yaml"
custom_tracker2 = "trackers/custom_botsort2.yaml"

In [6]:
### Directories
video_input_directory = Path("video/input")
video_output_directory = Path("video/output")
logs_frame_directory = Path("video/logs/logs_frames")
logs_summary_directory = Path("video/logs/logs_summary")

video_input_directory.mkdir(parents=True, exist_ok=True)
video_output_directory.mkdir(parents=True, exist_ok=True)
logs_frame_directory.mkdir(parents=True, exist_ok=True)
logs_summary_directory.mkdir(parents=True, exist_ok=True)

In [7]:
### Systems Configuration

##### Confidence Threshold
detect_conf_thres = 0.2
pose_conf_thres = 0.2

##### Video Stride
video_stride = 5

##### IoA Threshold
ioa_thres = 0.5

##### Frame Stitching Threshold
frame_stitching_threshold = 60

##### Live Display Settings
display_w = 800
display_h = 600
text_display_w = 400
text_display_h = 400

##### Progress
progress_frame_enabled = True
progress_bar_enabled = True
progress_percent_enabled = True
progress_elapsed_time_enabled = True
progress_bar_length = 40
progress_bar_character = '#'

##### Logging
live_summary_enablede = True

##### Colors
red = (255,0,0)
green = (0,255,0)
blue = (0,0,255)
white=(255,255,255)
black=(0,0,0)




pad = 15

In [8]:
### Clear Existing Output Directories
for item in video_output_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

for item in logs_frame_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

for item in logs_summary_directory.iterdir():
    if item.is_file() or item.is_symlink():
        item.unlink()
    elif item.is_dir():
        shutil.rmtree(item)

In [9]:
show_input_videos = True
if show_input_videos:
    for video_index, video in enumerate(video_input_directory.glob("*.mp4")):
        print(video.stem)

vid01
vid02
vid03
vid05
vid34


In [10]:
### Main Function
def process_video(video_path):

    ### Timer
    start_time = time.perf_counter()

    ### Extract Video Information
    video_name = video_path.stem

    ### Video Capture
    cap = cv2.VideoCapture(str(video_path))
    video_fps = int(cap.get(cv2.CAP_PROP_FPS))
    video_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    video_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) - 1 # Remove 1 for zero-indexing
    adjusted_video_fps = (video_fps / video_stride)


    ### Output Video
    output_video = video_output_directory / f"{video_name}_output.mp4"


    ### Writer
    fourcc = "mp4v" if using_colab else "avc1"
    writer = cv2.VideoWriter(
        str(output_video),
        cv2.VideoWriter_fourcc(*fourcc),
        video_fps,
        (video_width, video_height)
    )
    if not writer.isOpened():
        print(f"Failed to initialize VideoWriter for {output_video}")
        cap.release()
        return


    ### Output Logs
    hoid_log_path = logs_frame_directory / f"{video_name}_interaction_log.csv"
    hoid_log_summary_path = logs_summary_directory / f"{video_name}_summary_log.csv"
    hoid_log_readable_path = logs_frame_directory / f"{video_name}_interaction_readable_log.txt"
    hoid_log_summary_readable_path = logs_summary_directory / f"{video_name}_summary_readable_log.txt"


    ### Initialize Models
    detect_model = YOLO(detect_model_name)
    pose_model = YOLO(pose_model_name)
    #detect_classes = [key for key, value in detect_model.names.items() if value != "person"]
    detect_classes = [key for key, value in detect_model.names.items() if value == "phone"]
    pose_classes = [key for key, value in pose_model.names.items() if value == "person"]


    ### Create Log Files
    with (
        open(hoid_log_path, "w") as hoid_log,
        open(hoid_log_summary_path, "w") as hoid_log_summary,
        open(hoid_log_readable_path, "w") as hoid_log_readable,
        open(hoid_log_summary_readable_path, "w") as hoid_log_summary_readable
    ):
        hoid_log.write("frame_index,human_id,object_id\n")
        hoid_log_summary.write("frame_start,frame_end,human_id,object_id\n")



        ### Human-Object Interactions Summary
        interactions_summary = {}

        ### Main Loop
        frame_index = 0
        while cap.isOpened():

            success = cap.grab()
            if not success:
                break

            ### Human-Object Interactions
            interactions = []

            on_stride = frame_index % video_stride == 0
            if on_stride:

                success, frame = cap.retrieve()
                current_frame = frame

                ### Run Models on Current Frame
                ##### Detect Model
                detect_results = detect_model.track(
                    device=using_gpu,
                    source=current_frame,
                    tracker=custom_tracker2,
                    persist=True,
                    conf=detect_conf_thres,
                    verbose=False,
                    classes=detect_classes
                )
                ##### Pose Model
                pose_results = pose_model.track(
                    device=using_gpu,
                    source=current_frame,
                    tracker=custom_tracker,
                    persist=True,
                    conf=pose_conf_thres,
                    verbose=False,
                    classes=pose_classes
                )


                ### Model Results
                d_result = detect_results[0]
                p_result = pose_results[0]


                ### Extract Boxes from Detect and Pose Models
                d_boxes = d_result.boxes
                p_boxes = p_result.boxes


                ### Extract Detect Tensors
                d_xyxys = d_boxes.xyxy.clone()
                d_xyxys[:, :2] -= pad
                d_xyxys[:, 2:] += pad
                d_xyxys[:, [0, 2]] = d_xyxys[:, [0, 2]].clamp(min=0, max=video_width)
                d_xyxys[:, [1, 3]] = d_xyxys[:, [1, 3]].clamp(min=0, max=video_height)
                d_clss = d_boxes.cls.int()
                d_confs = d_boxes.conf
                d_track_ids = d_boxes.id.int() if d_boxes.id is not None else torch.zeros(len(d_boxes), dtype=torch.int32, device=d_xyxys.device)


                ### Extract Pose Tensors
                p_xyxys = p_boxes.xyxy.to(d_xyxys.device)
                p_confs = p_boxes.conf
                p_track_ids = p_boxes.id.int() if p_boxes.id is not None else torch.zeros(len(p_boxes), dtype=torch.int32, device=d_xyxys.device)
                kpts = p_result.keypoints.xy.to(d_xyxys.device) if p_result.keypoints is not None else None
            
                ### Human-Object Interaction Detection Block
                if len(d_boxes) > 0:
                    
                    ### Calculate All Object Bounding Areas
                    obj_area = (d_xyxys[:, 2] - d_xyxys[:, 0]) * (d_xyxys[:, 3] - d_xyxys[:, 1])


                    ### Calculate the Intersection Area 
                    xi1 = torch.maximum(p_xyxys[:, None, 0], d_xyxys[None, :, 0])
                    yi1 = torch.maximum(p_xyxys[:, None, 1], d_xyxys[None, :, 1])
                    xi2 = torch.minimum(p_xyxys[:, None, 2], d_xyxys[None, :, 2])
                    yi2 = torch.minimum(p_xyxys[:, None, 3], d_xyxys[None, :, 3])
                    inter_area = torch.clamp(xi2 - xi1, min=0) * torch.clamp(yi2 - yi1, min=0)               


                    ### Calculate Intersection-of-Area (IoA)
                    ioa = torch.zeros_like(inter_area)
                    valid_obj = obj_area > 0
                    ioa[:, valid_obj] = inter_area[:, valid_obj] / obj_area[valid_obj]


                    ### Filter Out Indices Below Threshold
                    p_indices, d_indices = torch.where(ioa >= ioa_thres)


                    ### Check For Interactions
                    if kpts is not None and len(p_indices) > 0:
                        matched_keypoints = kpts[p_indices][:, [9, 10]]
                        matched_boxes = d_xyxys[d_indices]
                        kx, ky = matched_keypoints[..., 0], matched_keypoints[..., 1]
                        bx1 = matched_boxes[:, None, 0]
                        by1 = matched_boxes[:, None, 1]
                        bx2 = matched_boxes[:, None, 2]
                        by2 = matched_boxes[:, None, 3]
                        keypoints_inside_box = (kx >= bx1) & (kx <= bx2) & (ky >= by1) & (ky <= by2)
                        has_interacting_keypoint = keypoints_inside_box.any(dim=1)
                        confirmed_person_indices = p_indices[has_interacting_keypoint].cpu()
                        confirmed_object_indices = d_indices[has_interacting_keypoint].cpu()
                        interactions.extend([
                            (
                                frame_index, 
                                p_track_ids[p], 
                                d_track_ids[d], 
                                detect_model.names[d_clss[d].item()]
                            )
                            for p, d in zip(confirmed_person_indices, confirmed_object_indices)
                        ])

                    ### End of Detection Block
                

                ### Plot Pose Keypoints
                current_frame = p_result.plot(boxes=False)


                ### Plot Pose Rects
                if len(p_boxes) > 0:
                    for i, (x1, y1, x2, y2) in enumerate(p_xyxys.int().tolist()):
                        cv2.rectangle(
                            img=current_frame,
                            pt1=(x1, y1),
                            pt2=(x2, y2),
                            color=blue,
                            thickness=2
                        )
                        cv2.putText(
                            img=current_frame,
                            text=f"ID:{p_track_ids[i]} CLS:person CONF:{round(p_confs[i].item(), 2)}",
                            org=(x1, y1 - 10),
                            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                            fontScale=1.0,
                            color=blue,
                            thickness=2
                        )

                    ## End of Drawing Loop Block
                

                ### Plot Detect Rects
                if len(d_boxes) > 0:

                    ### Fetch All IDs of Interacting Objects
                    interacting_obj_ids = [interaction[2] for interaction in interactions]
                    

                    ### Main Drawing Loop for Detect Rects
                    for i, (x1, y1, x2, y2) in enumerate(d_xyxys.int().tolist()):
                        ##### Determine If Object Is Interacting Or Not
                        interacting_with_human = d_track_ids[i] in interacting_obj_ids
                        ##### Draw Rect and Text
                        use_rgb = green if interacting_with_human else red
                        cv2.rectangle(
                            img=current_frame,
                            pt1=(x1, y1),
                            pt2=(x2, y2),
                            color=use_rgb,
                            thickness=2
                        )
                        cv2.putText(
                            img=current_frame,
                            text=f"ID:{d_track_ids[i]} CLS:{detect_model.names[d_clss[i].item()]} CONF:{round(d_confs[i].item(), 2)}",
                            org=(x1, y1 - 10),
                            fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                            fontScale=1.0,
                            color=red,
                            thickness=2
                        )
                        ##### Draw Interaction Alert Text
                        if interacting_with_human:
                            interacting_humans = [inter[1].item() for inter in interactions if inter[2] == d_track_ids[i]]
                            humans_str = ",".join(map(str, interacting_humans))
                            cv2.putText(
                                img=current_frame,
                                text=f"Human:{humans_str} Object:{d_track_ids[i]}",
                                org=(x1, y1 + 25),
                                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                                fontScale=1.0,
                                color=green,
                                thickness=2
                            )
                        
                    ### End of Drawing Loop Block


                ### Write Interactions to Log File
                if interactions:
                    hoid_log.writelines([
                        f"{f_index},{per_id},{o_id}\n"
                        for f_index, per_id, o_id, _ in interactions
                    ])
                    hoid_log_readable.writelines([
                        f"FRAME {frame_index:6d} : PERSON {per_id:4d} and {obj_class} {o_id:4d}\n"
                        for f_index, per_id, o_id, obj_class in interactions
                    ])
                
                ### End of If On-Stride Block


            ### Generating Interactions Summary
            for f_index, per_id, o_id, _ in interactions:
                key = (per_id.item(), o_id.item())
                if key not in interactions_summary:
                    interactions_summary[key] = [[f_index, f_index]]
                elif f_index - interactions_summary[key][-1][1] <= frame_stitching_threshold:
                    interactions_summary[key][-1][1] = f_index
                else:
                    interactions_summary[key].append([f_index, f_index])


            ### Frame Writing
            output_frame = current_frame.copy()
            cv2.putText(
                img=output_frame,
                text=f"Frame {frame_index}",
                org=(30, 40),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                fontScale=1.0,
                color=white,
                thickness=3
            )
            writer.write(output_frame)


            ### Live Video Display
            liev_video_window_title = f"{vid} Live Video" 
            cv2.imshow('liev_video_window_title', output_frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break


            ### Timestamp
            est_time_seconds = int(frame_index / video_fps)
            minutes, seconds = divmod(est_time_seconds, 60)
            timestamp = f"{minutes:02d}:{seconds:02d}"


            ### Progress Report
            ##### Progress Frame
            progress_frame = ""
            if progress_frame_enabled:
                progress_frame = f"{frame_index}/{total_frames}"
            ##### Progress Bar
            progress_bar = ""
            if progress_bar_enabled:
                filled = int(progress_bar_length * (frame_index) // total_frames)
                bar = '#' * filled + '-' * (progress_bar_length - filled)
                progress_bar = f"[{bar}]"
            ##### Progress Percent
            progress_percent = ""
            if progress_percent_enabled:
                percent = (frame_index / total_frames) * 100
                progress_percent = f"{percent:.2f}%"
            ##### Progress Time
            progress_elapsed_time = ""
            if progress_elapsed_time_enabled:
                elapsed_time = time.perf_counter() - start_time
                progress_elapsed_time = f"{elapsed_time:.2f}s"
            ##### Print Progress
            print(f"\r{video_name:<7} {progress_frame:13} {progress_bar} {progress_percent:7} {progress_elapsed_time:7}", end="", flush=True)

            frame_index = frame_index + 1
    
            ### End of While-Block


        ### Write Interactions Summary To Log Summary File
        if interactions_summary:
            hoid_log_summary.writelines([ 
                f"{f1},{f2},{k1},{k2}\n"
                for (k1, k2), v in interactions_summary.items()
                for f1, f2 in v if (f2 - f1) >= 30
            ])
            hoid_log_summary_readable.writelines([
                item 
                for (k1, k2), v in interactions_summary.items() if any((f2 - f1) > 30 for f1, f2 in v)
                for item in [f"Human {k1} and Object {k2}\n"] + [f"    Frame {f1:5} to Frame {f2:5}\n" for f1, f2 in v if (f2 - f1) > 30]
            ])

    cap.release()
    writer.release()
    cv2.destroyAllWindows()
    print()

In [11]:
# print(f"Begin")
# for video in video_input_directory.glob("*.mp4"):
#     process_video(video)
# print(f"\nAll Done\n")

In [12]:
vid_id_list = [1]
vid_name_list = [f"vid{vid:02d}" for vid in vid_id_list]
vid_path_list = [Path(f"video/input/{name}.mp4") for name in vid_name_list]

print(f"Begin")
for video in vid_path_list:
    # print(video)
    process_video(video)
print(f"All Done")

Begin


FileNotFoundError: 'trackers/custom_botsort2.yaml' does not exist